# Multi-turn Conversation Management

Building chat applications with Claude requires managing conversation history effectively. As conversations grow, you'll hit context limits, increase costs, and slow down responses.

This cookbook teaches you practical strategies for managing multi-turn conversations:

- **Understanding context limits** and token counting
- **Sliding window** approaches for simple truncation
- **Conversation summarization** to preserve context while reducing tokens
- **Hybrid strategies** that combine both techniques

**Related:** For advanced background compaction with threading, see [Session Memory Compaction](./session_memory_compaction.ipynb). For SDK-based automatic compaction in agentic workflows, see [Automatic Context Compaction](../tool_use/automatic-context-compaction.ipynb).

## Setup

In [ ]:
%pip install anthropic python-dotenv --quiet

In [ ]:
import anthropic
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()

MODEL_NAME = "claude-sonnet-4-6"

---
## How Multi-turn Conversations Work

Claude's Messages API is **stateless** — each request must include the full conversation history. The API doesn't remember previous calls.

```python
# Each request includes ALL previous messages
messages = [
    {"role": "user", "content": "Hi, I'm Alice."},
    {"role": "assistant", "content": "Hello Alice! How can I help?"},
    {"role": "user", "content": "What's my name?"},  # Claude sees full history
]
```

This means **you control the context**. You decide what history to include, what to drop, and what to summarize.

In [ ]:
# Basic multi-turn example
messages = []


def chat(user_message: str) -> str:
    """Send a message and get a response, maintaining conversation history."""
    messages.append({"role": "user", "content": user_message})

    response = client.messages.create(model=MODEL_NAME, max_tokens=1024, messages=messages)

    assistant_message = response.content[0].text
    messages.append({"role": "assistant", "content": assistant_message})

    return assistant_message


# Demonstrate memory across turns
print("Turn 1:", chat("Hi, my name is Alice and I'm a software engineer."))
print("\nTurn 2:", chat("What's my profession?"))
print("\nTurn 3:", chat("And my name?"))

---
## Understanding Context Limits

Each Claude model has a **context window** — the maximum number of tokens it can process (input + output combined).

| Model | Context Window |
|-------|---------------|
| Claude Opus 4.6 | 200K tokens |
| Claude Sonnet 4.6 | 200K tokens |
| Claude Haiku 4.5 | 200K tokens |

**Why this matters:**
- Longer conversations = more tokens = higher cost
- Approaching limits = potential truncation or errors
- More context = slower time-to-first-token

### Token Counting

Use the `count_tokens` endpoint to measure your conversation before sending it.

In [ ]:
def count_conversation_tokens(messages: list, system: str = None) -> int:
    """Count tokens in a conversation using the API."""
    params = {
        "model": MODEL_NAME,
        "messages": messages,
    }
    if system:
        params["system"] = system

    response = client.messages.count_tokens(**params)
    return response.input_tokens


# Count tokens in our conversation so far
token_count = count_conversation_tokens(messages)
print(f"Current conversation: {token_count} tokens")
print(f"Messages in history: {len(messages)}")

In [ ]:
# Quick estimation without API call (for rough planning)
def estimate_tokens(text: str) -> int:
    """Rough estimate: ~4 characters per token for English text."""
    return len(text) // 4


sample_text = "This is a sample message to estimate token count."
print(f"Estimated: ~{estimate_tokens(sample_text)} tokens")
print("Note: Use count_tokens() for accurate counts before critical decisions.")

---
## Strategy 1: Sliding Window

The simplest approach: keep only the **N most recent messages**. Old messages are dropped.

**Pros:** Simple, predictable token usage, fast

**Cons:** Loses information from earlier in the conversation

In [ ]:
class SlidingWindowChat:
    """Chat that keeps only the last N message pairs."""

    def __init__(self, max_pairs: int = 10, system: str = None):
        self.messages = []
        self.max_pairs = max_pairs  # Each pair = 1 user + 1 assistant message
        self.system = system

    def _truncate(self):
        """Keep only the last max_pairs conversation pairs."""
        max_messages = self.max_pairs * 2
        if len(self.messages) > max_messages:
            self.messages = self.messages[-max_messages:]

    def chat(self, user_message: str) -> str:
        self.messages.append({"role": "user", "content": user_message})

        response = client.messages.create(
            model=MODEL_NAME, max_tokens=1024, system=self.system or "", messages=self.messages
        )

        assistant_message = response.content[0].text
        self.messages.append({"role": "assistant", "content": assistant_message})

        # Truncate after adding the response
        self._truncate()

        return assistant_message

    def get_stats(self) -> dict:
        return {"message_count": len(self.messages), "max_messages": self.max_pairs * 2}

In [ ]:
# Demo: sliding window with max 3 pairs (6 messages)
sw_chat = SlidingWindowChat(max_pairs=3)

conversations = [
    "My name is Bob.",
    "I live in Tokyo.",
    "I work as a chef.",
    "My favorite dish is ramen.",
    "What do you know about me?",  # Will only remember recent context
]

for msg in conversations:
    print(f"User: {msg}")
    response = sw_chat.chat(msg)
    print(f"Claude: {response}")
    print(f"Stats: {sw_chat.get_stats()}")
    print("-" * 50)

### Token-based Sliding Window

Instead of counting messages, limit by **token count** for more precise control.

In [ ]:
class TokenLimitedChat:
    """Chat that keeps messages under a token limit."""

    def __init__(self, max_tokens: int = 4000, system: str = None):
        self.messages = []
        self.max_tokens = max_tokens
        self.system = system

    def _get_token_count(self) -> int:
        """Get current token count via API."""
        if not self.messages:
            return 0
        return count_conversation_tokens(self.messages, self.system)

    def _truncate_to_limit(self):
        """Remove oldest messages until under token limit."""
        while len(self.messages) > 2 and self._get_token_count() > self.max_tokens:
            # Remove oldest pair (user + assistant)
            self.messages = self.messages[2:]

    def chat(self, user_message: str) -> str:
        self.messages.append({"role": "user", "content": user_message})

        response = client.messages.create(
            model=MODEL_NAME, max_tokens=1024, system=self.system or "", messages=self.messages
        )

        assistant_message = response.content[0].text
        self.messages.append({"role": "assistant", "content": assistant_message})

        self._truncate_to_limit()

        return assistant_message, self._get_token_count()


# Demo with low token limit to force truncation
token_chat = TokenLimitedChat(max_tokens=500)
response, tokens = token_chat.chat("Tell me a short story about a brave knight.")
print(f"Response: {response[:200]}...")
print(f"Current tokens: {tokens}")

---
## Strategy 2: Conversation Summarization

Use Claude to **summarize older messages**, preserving key information in fewer tokens.

**Pros:** Retains important context from entire conversation

**Cons:** Extra API call for summarization, some nuance may be lost

In [ ]:
SUMMARY_PROMPT = """Summarize the following conversation excerpt concisely.
Focus on: key facts mentioned, user preferences, decisions made, and important context.
Keep it under 200 words.

Conversation:
{conversation}

Summary:"""


def summarize_messages(messages: list) -> str:
    """Generate a summary of the given messages."""
    # Format messages as readable conversation
    conversation_text = "\n".join(f"{msg['role'].title()}: {msg['content']}" for msg in messages)

    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=300,
        messages=[
            {"role": "user", "content": SUMMARY_PROMPT.format(conversation=conversation_text)}
        ],
    )

    return response.content[0].text

In [ ]:
class SummarizingChat:
    """Chat that summarizes older messages to stay under token limits."""

    def __init__(self, max_tokens: int = 2000, keep_recent: int = 4, system: str = None):
        self.messages = []
        self.summary = None
        self.max_tokens = max_tokens
        self.keep_recent = keep_recent  # Keep this many recent messages unsummarized
        self.system = system

    def _get_system_with_summary(self) -> str:
        """Combine system prompt with conversation summary."""
        parts = []
        if self.system:
            parts.append(self.system)
        if self.summary:
            parts.append(f"\n\nPrevious conversation summary:\n{self.summary}")
        return "\n".join(parts) if parts else ""

    def _maybe_summarize(self):
        """Summarize old messages if over token limit."""
        current_tokens = count_conversation_tokens(self.messages, self._get_system_with_summary())

        if current_tokens > self.max_tokens and len(self.messages) > self.keep_recent:
            # Split messages: old ones to summarize, recent ones to keep
            old_messages = self.messages[: -self.keep_recent]
            recent_messages = self.messages[-self.keep_recent :]

            # Generate new summary combining old summary + old messages
            to_summarize = old_messages
            if self.summary:
                to_summarize = [
                    {"role": "system", "content": f"Previous summary: {self.summary}"}
                ] + old_messages

            self.summary = summarize_messages(to_summarize)
            self.messages = recent_messages

            print(f"[Summarized {len(old_messages)} messages]")

    def chat(self, user_message: str) -> str:
        self.messages.append({"role": "user", "content": user_message})

        response = client.messages.create(
            model=MODEL_NAME,
            max_tokens=1024,
            system=self._get_system_with_summary(),
            messages=self.messages,
        )

        assistant_message = response.content[0].text
        self.messages.append({"role": "assistant", "content": assistant_message})

        self._maybe_summarize()

        return assistant_message

    def get_stats(self) -> dict:
        return {
            "messages": len(self.messages),
            "has_summary": self.summary is not None,
            "tokens": count_conversation_tokens(self.messages, self._get_system_with_summary()),
        }

In [ ]:
# Demo: summarizing chat with low limits to trigger summarization
sum_chat = SummarizingChat(max_tokens=1000, keep_recent=4)

test_messages = [
    "Hi! I'm planning a trip to Japan next month.",
    "I'll be staying for 2 weeks, mostly in Tokyo and Kyoto.",
    "I'm vegetarian and love traditional architecture.",
    "What temples should I visit in Kyoto?",
    "Also, I'm traveling with my partner who loves food markets.",
    "We're on a moderate budget, around $150/day for both of us.",
    "Can you remind me what my dietary restriction is?",  # Tests memory
]

for msg in test_messages:
    print(f"\nUser: {msg}")
    response = sum_chat.chat(msg)
    print(f"Claude: {response[:300]}{'...' if len(response) > 300 else ''}")
    print(f"Stats: {sum_chat.get_stats()}")

In [ ]:
# View the summary that was created
if sum_chat.summary:
    print("Generated Summary:")
    print(sum_chat.summary)

---
## Strategy 3: Hybrid Approach

Combine multiple strategies for robust conversation management:

1. **Always keep** the system prompt and critical context
2. **Summarize** periodically to capture long-term context
3. **Keep recent messages** verbatim for immediate context
4. **Use prompt caching** to reduce costs on stable prefixes

In [ ]:
class HybridConversationManager:
    """
    Production-ready conversation manager combining:
    - Persistent system prompt
    - Rolling summary of old context
    - Recent message window
    - Token budget awareness
    """

    def __init__(
        self,
        system_prompt: str,
        max_context_tokens: int = 8000,
        recent_window: int = 6,
        summarize_threshold: int = 10,
    ):
        self.system_prompt = system_prompt
        self.max_context_tokens = max_context_tokens
        self.recent_window = recent_window
        self.summarize_threshold = summarize_threshold

        self.messages = []
        self.summary = ""
        self.total_messages_processed = 0

    def _build_system(self) -> str:
        """Construct full system prompt with summary context."""
        if self.summary:
            return f"{self.system_prompt}\n\n<conversation_history_summary>\n{self.summary}\n</conversation_history_summary>"
        return self.system_prompt

    def _should_summarize(self) -> bool:
        """Check if we should trigger summarization."""
        token_count = count_conversation_tokens(self.messages, self._build_system())
        message_count = len(self.messages)

        return (
            token_count > self.max_context_tokens * 0.8 or message_count > self.summarize_threshold
        )

    def _perform_summarization(self):
        """Summarize older messages and update state."""
        if len(self.messages) <= self.recent_window:
            return

        old_messages = self.messages[: -self.recent_window]

        # Include existing summary in what we're summarizing
        context_to_summarize = []
        if self.summary:
            context_to_summarize.append(
                {"role": "assistant", "content": f"[Prior context: {self.summary}]"}
            )
        context_to_summarize.extend(old_messages)

        self.summary = summarize_messages(context_to_summarize)
        self.messages = self.messages[-self.recent_window :]

        print(f"[Compacted: summarized {len(old_messages)} messages]")

    def send_message(self, user_message: str) -> str:
        """Send a message and get response with automatic context management."""
        self.messages.append({"role": "user", "content": user_message})
        self.total_messages_processed += 1

        # Check if we need to summarize BEFORE the API call
        if self._should_summarize():
            self._perform_summarization()

        response = client.messages.create(
            model=MODEL_NAME, max_tokens=1024, system=self._build_system(), messages=self.messages
        )

        assistant_message = response.content[0].text
        self.messages.append({"role": "assistant", "content": assistant_message})
        self.total_messages_processed += 1

        return assistant_message

    def get_status(self) -> dict:
        """Get current conversation status."""
        return {
            "active_messages": len(self.messages),
            "total_processed": self.total_messages_processed,
            "has_summary": bool(self.summary),
            "current_tokens": count_conversation_tokens(self.messages, self._build_system()),
        }

In [ ]:
# Demo: Hybrid approach with a customer service scenario
system = """You are a helpful customer service agent for TechCorp.
Be concise and friendly. Remember customer details mentioned in the conversation."""

manager = HybridConversationManager(
    system_prompt=system,
    max_context_tokens=2000,  # Low for demo
    recent_window=4,
    summarize_threshold=8,
)

customer_messages = [
    "Hi, I'm having issues with my TechCorp Pro subscription. My account email is alex@example.com",
    "The problem is that I can't access the premium features even though I'm paying for them.",
    "I've been a customer for 3 years and this is really frustrating.",
    "I tried logging out and back in but it didn't help.",
    "Can you check my account status?",
    "Also, I wanted to mention I'm on the annual plan.",
    "What's my email address again?",  # Tests if context is preserved
]

for msg in customer_messages:
    print(f"\nCustomer: {msg}")
    response = manager.send_message(msg)
    print(f"Agent: {response}")
    print(f"Status: {manager.get_status()}")

---
## Best Practices

### 1. Choose the Right Strategy

| Use Case | Recommended Strategy |
|----------|---------------------|
| Simple chatbot, short conversations | Sliding window |
| Customer service, need full context | Summarization |
| Long-running agents | Hybrid approach |
| Code assistance | Keep recent code, summarize discussions |

### 2. Optimize System Prompts

Your system prompt is sent with every request. Keep it focused.

In [ ]:
# Compare system prompt sizes
verbose_system = """
You are an incredibly helpful and friendly AI assistant. Your primary goal is to assist
users with their questions and tasks in the most helpful way possible. You should always
strive to be accurate, clear, and concise in your responses. Remember to be polite and
professional at all times. If you don't know something, admit it honestly rather than
making up information. Always think step by step when solving problems.
"""

concise_system = """Helpful AI assistant. Be accurate, clear, concise. Admit uncertainty."""

print(f"Verbose system: ~{estimate_tokens(verbose_system)} tokens")
print(f"Concise system: ~{estimate_tokens(concise_system)} tokens")
print(
    f"Savings per request: ~{estimate_tokens(verbose_system) - estimate_tokens(concise_system)} tokens"
)

### 3. Use Prompt Caching for Long Conversations

When using summarization, your system prompt + summary stays stable. Use prompt caching to reduce costs.

```python
response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=1024,
    cache_control={"type": "ephemeral"},  # Enable automatic caching
    system=system_with_summary,
    messages=recent_messages
)
```

### 4. Handle Edge Cases

In [ ]:
def safe_chat(messages: list, user_message: str, max_retries: int = 2) -> str:
    """
    Chat with automatic truncation on context length errors.
    """
    messages_copy = messages.copy()
    messages_copy.append({"role": "user", "content": user_message})

    for _attempt in range(max_retries):
        try:
            response = client.messages.create(
                model=MODEL_NAME, max_tokens=1024, messages=messages_copy
            )
            return response.content[0].text

        except anthropic.BadRequestError as e:
            if "context length" in str(e).lower() or "too long" in str(e).lower():
                # Emergency truncation: remove oldest half of messages
                mid = len(messages_copy) // 2
                messages_copy = messages_copy[mid:]
                print(f"[Context too long, truncated to {len(messages_copy)} messages]")
            else:
                raise

    raise RuntimeError("Failed after max retries")


# This pattern catches context length errors and recovers gracefully
print("Safe chat function defined - handles context overflow automatically")

---
## Summary

| Strategy | Best For | Token Efficiency | Context Preservation |
|----------|----------|------------------|---------------------|
| **Sliding Window** | Short chats, simple use cases | High | Low (loses old context) |
| **Summarization** | Support chats, need history | Medium | High (compressed) |
| **Hybrid** | Production apps, long sessions | Configurable | High |

**Key takeaways:**
1. You control the conversation history — Claude is stateless
2. Monitor token usage with `count_tokens()`
3. Summarize to preserve context while reducing tokens
4. Keep recent messages verbatim for immediate context
5. Use prompt caching when your prefix is stable

For more advanced patterns with background processing, see:
- [Session Memory Compaction](./session_memory_compaction.ipynb)
- [Automatic Context Compaction](../tool_use/automatic-context-compaction.ipynb)